# Rice paddy flooding from Sentinel-1: exploration

**Goal.** Detect, for every rice pixel and every Sentinel-1 date, whether the field is **flooded or drained**. Counting drainage episodes during the season gives a proxy for **Alternate Wetting and Drying (AWD)**, the practice that cuts rice methane emissions by about half.

**Why radar works.** Calm open water acts as a mirror: the radar pulse bounces away and almost nothing comes back, so backscatter is very low (VH below about -20 dB). As the rice grows, stems and leaves scatter more energy back and VH rises. Once the canopy is dense, flooded rice can even give a *higher* VV signal (double bounce between water surface and stems), which is the hard case.

**Labels.** Sentinel-2 sees water clearly (MNDWI index) but only when there are no clouds. We use clear-sky Sentinel-2 pixels acquired within a day of a Sentinel-1 image as weak labels, and train a radar model that also works under clouds.

Set `USE_SYNTHETIC = False` once `scripts/01_download.py` and `scripts/02_make_labels.py` have run.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from ricepaddy.config import load_config
from ricepaddy.data import open_s1
from ricepaddy.features import build_features, rice_mask_heuristic, to_table

warnings.filterwarnings("ignore")
CONFIG = None   # None = chapter 1 (config.yaml); "../config_ch2.yaml" = both orbits
cfg = load_config(CONFIG)
USE_SYNTHETIC = not Path(cfg["paths"]["labels"]).exists()

if USE_SYNTHETIC:
    from ricepaddy.synthetic import make_synthetic
    s1, labels, truth = make_synthetic()
    print("Using SYNTHETIC data (run scripts 01 and 02 to use real Sentinel data)")
else:
    s1 = open_s1(cfg)   # normalised per orbit if the config says so
    labels = xr.open_zarr(cfg["paths"]["labels"])["label"].load()
s1

## 1. What the scene looks like
Mean VH over the season, and a false-colour composite at one date (R = VV, G = VH, B = VV - VH). Field boundaries and canals should be visible.

In [ ]:
def stretch(a, lo=2, hi=98):
    p1, p2 = np.nanpercentile(a, [lo, hi])
    return np.clip((a - p1) / (p2 - p1), 0, 1)

t = s1.sizes["time"] // 2
rgb = np.dstack([stretch(s1.vv_db[t].values), stretch(s1.vh_db[t].values),
                 stretch((s1.vv_db[t] - s1.vh_db[t]).values)])
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
s1.vh_db.mean("time").plot(ax=axs[0], cmap="gray"); axs[0].set_title("Mean VH (dB)")
axs[1].imshow(rgb); axs[1].set_title(f"R=VV G=VH B=VV-VH, {str(s1.time.values[t])[:10]}")
for a in axs: a.set_aspect("equal")
plt.tight_layout()

## 2. How many labels do the clouds leave us?
Share of pixels with a clear-sky Sentinel-2 label on each Sentinel-1 date.

In [ ]:
cov = (labels >= 0).mean(["y", "x"]).to_series()
water_share = ((labels == 1).sum(["y", "x"]) / (labels >= 0).sum(["y", "x"])).to_series()
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(cov.index, cov.values, width=3, label="labelled share")
ax.plot(water_share.index, water_share.values, "o-", c="tab:blue", label="water share among labelled")
ax.set_ylim(0, 1); ax.legend(); ax.set_title("Label coverage per Sentinel-1 date")

## 3. Time series of a few pixels
Backscatter through the season with the weak labels on top (blue = water, brown = dry). Look for the drop when fields are flooded and the rise as the crop grows.

In [ ]:
rng = np.random.default_rng(1)
pix = [(rng.integers(s1.sizes["y"]), rng.integers(s1.sizes["x"])) for _ in range(4)]
fig, axs = plt.subplots(len(pix), 1, figsize=(11, 2.4 * len(pix)), sharex=True)
for ax, (iy, ix) in zip(axs, pix):
    ts = s1.isel(y=iy, x=ix)
    ax.plot(ts.time, ts.vh_db, "o-", label="VH"); ax.plot(ts.time, ts.vv_db, "s-", alpha=.6, label="VV")
    lab = labels.isel(y=iy, x=ix).values
    for tt, l in zip(ts.time.values, lab):
        if l >= 0:
            ax.axvspan(tt - np.timedelta64(2, "D"), tt + np.timedelta64(2, "D"),
                       color="tab:blue" if l == 1 else "sienna", alpha=.25)
    ax.set_ylabel("dB"); ax.set_title(f"pixel y={iy}, x={ix}", fontsize=9)
axs[0].legend(loc="upper left")
plt.tight_layout()

## 4. Can a single threshold separate water from dry soil?
Histogram of VH for labelled water vs dry pixel-dates. The overlap is what the random forest (and later a deep model) must resolve, mostly with temporal context.

In [ ]:
vh = s1.vh_db.values; lab = labels.values
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(-30, -5, 80)
ax.hist(vh[lab == 0], bins, alpha=.6, density=True, label="dry (S2)", color="sienna")
ax.hist(vh[lab == 1], bins, alpha=.6, density=True, label="water (S2)", color="tab:blue")
ax.set_xlabel("VH (dB)"); ax.legend()

## 5. Rice mask (heuristic)
Pixels flooded at least once and with a strong VH rise over the season. Compare with Google satellite view of the area to sanity-check: villages, canals and orchards should be excluded.

In [ ]:
rice = rice_mask_heuristic(s1)
rice.plot(cmap="Greens", figsize=(6, 5)); plt.gca().set_aspect("equal")
print(f"rice share: {float(rice.mean()):.2f}")

## 6. Cross-validation: how much does a random split flatter the model?
Results written by `scripts/03_baseline.py`. Neighbouring pixels of the same field are near-duplicates: a random split tests the model on fields it has already seen. Spatial blocks test on new areas, temporal splits on later dates (the crop has changed). The gap between *random* and *spatial_blocks* measures leakage.

In [ ]:
res_path = Path(cfg["paths"]["outputs"]) / ("synthetic" if USE_SYNTHETIC else cfg.get("run_name", "real")) / "cv_results.csv"
if res_path.exists():
    res = pd.read_csv(res_path)
    piv = res.groupby(["model", "scheme"])["balanced_accuracy"].agg(["mean", "std"]).unstack("scheme")
    piv["mean"].plot.bar(yerr=piv["std"], figsize=(8, 3.5), rot=0, ylim=(0.4, 1), capsize=3)
    plt.ylabel("balanced accuracy"); plt.title("Baselines under three cross-validation schemes")
else:
    print("Run: python scripts/03_baseline.py" + (" --synthetic" if USE_SYNTHETIC else ""))

## 7. AWD indicators
Predicted number of drying episodes per pixel during the season. Continuous-flooding fields should show 0 or 1, AWD fields 2 or more.

In [ ]:
pred_path = res_path.parent / "predictions.zarr"
if pred_path.exists():
    pred = xr.open_zarr(pred_path).load()
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    pred.drying_events.plot(ax=axs[0], cmap="viridis"); axs[0].set_title("Drying events")
    pred.flooded_fraction.plot(ax=axs[1], cmap="Blues", vmin=0, vmax=1); axs[1].set_title("Fraction of dates flooded")
    for a in axs: a.set_aspect("equal")
    plt.tight_layout()

## 8. Questions to dig into
- **Label noise**: MNDWI misses water under a dense canopy. Do late-season "dry" labels look suspicious in the time series?
- **Double bounce**: does VV go *up* on flooded fields late in the season? How does the random forest cope?
- **Leakage**: how big is the random vs spatial gap on real data? What block size makes it disappear?
- **Next step (PyTorch)**: a small temporal model (1D CNN or GRU over the whole VH/VV time series of a pixel), pre-trained self-supervised on all unlabelled pixels (masked time-step reconstruction), then fine-tuned on the weak labels. Evaluate with spatial blocks.